### ***SUMMARY***

Framing divides each audio signal into short, fixed-size sections that overlap. Speech changes over time, but it is nearly stable within each short frame. Feature methods such as magnitude spectra, MFCC, LPCC, and PLP can therefore describe each frame consistently. Fixed-size frames also give the classifier samples with the same dimensions.

The saved NumPy archive stores audio frames as `float32`, the rapper label as `bool`, frame numbers as `int32`, and path encodings as `uint8`. The fitted path encoder is saved separately as a joblib file so encoded paths can be mapped back to their source files. Keeping source-file links also supports data grouping and helps prevent leakage between training and test sets.

Saving the framed data makes preprocessing repeatable and avoids framing every file again during feature extraction. Later notebooks can load one consistent data set, apply windowing and feature extraction, and compare methods on the same frames.

In [ ]:
from pathlib import Path

# In a notebook, Path.cwd() is the folder the notebook runs from.
NOTEBOOK_DIR:Path = Path.cwd()
REPOSITORY_ROOT:Path = NOTEBOOK_DIR

DIR_SOURCE:Path = REPOSITORY_ROOT / "Artists"
DIR_OUTPUT:Path = NOTEBOOK_DIR / "preprocessing-01-framing.npz"
DIR_OUTPUT_ENCODER:Path = NOTEBOOK_DIR / "preprocessing-01-path-encoder.joblib"

if not DIR_SOURCE.is_dir():
    raise FileNotFoundError(f"Source directory does not exist: {DIR_SOURCE=}")

# =========================
# EXAMPLE
# =========================
print(f"Source directory: {DIR_SOURCE}")
print(f"Output file: {DIR_OUTPUT}")
print(f"Output file: {DIR_OUTPUT_ENCODER}")

### ***GETTING FILE DIRECTORIES***

In [ ]:
AUDIO_SUFFIXES:set[str] = {".flac", ".m4a", ".mp3", ".ogg", ".wav"}

DIR_SOURCE:Path = DIR_SOURCE.resolve()
dir_sources:list[Path] = sorted(
    path
    for path in DIR_SOURCE.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES
)
if not dir_sources:
    raise FileNotFoundError(f"No supported audio dir_sources found under {DIR_SOURCE}")

import numpy as np
from sklearn.preprocessing import LabelEncoder

# turn absolute paths into relative path strings
PATHS:list[str] = [audio_path.relative_to(DIR_SOURCE).as_posix() for audio_path in dir_sources]

# The encoder maps each relative source path to a compact integer code.
path_encoder:LabelEncoder = LabelEncoder()
dir_sources_encoded:np.ndarray = path_encoder.fit_transform(PATHS)
dir_sources_encoded = dir_sources_encoded.astype(np.uint8)

# warning check
if len(path_encoder.classes_) > np.iinfo(np.uint8).max + 1:
    raise OverflowError("More than 256 source paths cannot be encoded as uint8")

for x in dir_sources_encoded:
    print(f"{x:3} = {path_encoder.inverse_transform([x])}")

### ***FRAMING THE AUDIO FILES***

`sliding_window_view` creates a reference to the original audio 
- on frame[i] it returns audio [i to i+FRAME_SIZE]
- [::STEP_SIZE] is native python functionality which extracts only very other entry in a list
    - [::3] would return entry of n = 0, 3, 6, 9

In [ ]:
import librosa


SAMPLE_RATE:int = 16_000
# 0.025 seconds
FRAME_SIZE:int = 400       
# 0.010 seconds
STEP_SIZE:int = 160         

def frame_audio(file_audio: Path) -> np.ndarray:
    audio:np.ndarray
    audio, _ = librosa.load(file_audio, sr=SAMPLE_RATE, mono=True, dtype=np.float32)

    if audio.size < FRAME_SIZE:
        return np.empty((0, FRAME_SIZE), dtype=np.float32)

    frames:np.ndarray = np.lib.stride_tricks.sliding_window_view(audio, FRAME_SIZE)[::STEP_SIZE]
    return frames.astype(np.float32, copy=True)

# =========================
# EXAMPLE
# =========================

print(dir_sources[42])
print(frame_audio(dir_sources[42])[42])

In [ ]:
frame_blocks:list[np.ndarray] = []
frame_y_block:list[bool] = []
frame_number_block:list[np.int32] = []
frame_path_encoding_block:list[np.uint8] = []

for path_id, audio_dir in enumerate(dir_sources):
    frames:np.ndarray = frame_audio(audio_dir)
    frame_blocks.append(frames)

    is_rapper:bool = audio_dir.parent.name.lower() == "rapper"
    frame_y_block.extend(
        [is_rapper] * len(frames
    ))
    frame_number_block.extend(
        np.arange(1, len(frames) + 1, dtype=np.int32)
    )
    frame_path_encoding_block.extend(
        [np.uint8(dir_sources_encoded[path_id])] * len(frames)
    )
    print(f"{len(frames)} frames : {audio_dir}")


ALL_AUDIO_FRAMES:np.ndarray     = np.concatenate(frame_blocks, axis=0).astype(np.float32, copy=False)
FRAME_Y:np.ndarray              = np.asarray(frame_y_block,             dtype=np.bool_)
FRAME_NUMBERS:np.ndarray        = np.asarray(frame_number_block,        dtype=np.int32)
FRAME_PATH_ENCODING:np.ndarray  = np.asarray(frame_path_encoding_block, dtype=np.uint8)

### ***SAVING THE AUDIO FRAME FILES***

In [ ]:
if DIR_OUTPUT.suffix.lower() != ".npz":
    raise ValueError(f"Output file must use the .npz extension: {DIR_OUTPUT}")

DIR_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    DIR_OUTPUT,
    frame=ALL_AUDIO_FRAMES.astype(np.float32, copy=False),
    frame_y=FRAME_Y.astype(np.bool_, copy=False),
    frame_number=FRAME_NUMBERS.astype(np.int32, copy=False),
    frame_path_encoding=FRAME_PATH_ENCODING.astype(np.uint8, copy=False),
)

import joblib
joblib.dump(path_encoder, DIR_OUTPUT_ENCODER)

print(f"Saved {len(ALL_AUDIO_FRAMES)} frames to {DIR_OUTPUT}")
print(f"Saved path encoder to {DIR_OUTPUT_ENCODER}")
print("Members: FRAME (float32), FRAME_Y (bool), frame_number (int32), frame_path_encoding (uint8)")

### ***VERIFYING THE SAVED AUDIO FRAMES***

In [ ]:
with np.load(DIR_OUTPUT, allow_pickle=False) as saved:
    loaded_frames:np.ndarray = saved["frame"]
    loaded_frame_y:np.ndarray = saved["frame_y"]
    loaded_frame_numbers:np.ndarray = saved["frame_number"]
    loaded_frame_path_encoding:np.ndarray = saved["frame_path_encoding"]

assert loaded_frames.dtype == np.float32
assert loaded_frame_y.dtype == np.bool_
assert loaded_frame_numbers.dtype == np.int32
assert loaded_frame_path_encoding.dtype == np.uint8
assert np.array_equal(loaded_frames, ALL_AUDIO_FRAMES), (
    "One or more audio frames changed after saving."
)
assert np.array_equal(loaded_frame_y, FRAME_Y), "One or more labels changed after saving."
assert np.array_equal(loaded_frame_numbers, FRAME_NUMBERS), (
    "One or more frame numbers changed after saving."
)
assert np.array_equal(loaded_frame_path_encoding, FRAME_PATH_ENCODING), (
    "One or more encoded paths changed after saving."
)
assert len(loaded_frames) == len(loaded_frame_y) == len(loaded_frame_numbers) == len(loaded_frame_path_encoding)
assert int(np.max(loaded_frame_path_encoding, initial=0)) < len(path_encoder.classes_)

for path_code, relative_path in enumerate(path_encoder.classes_):
    frame_numbers = loaded_frame_numbers[loaded_frame_path_encoding == path_code]
    assert np.array_equal(
        frame_numbers, np.arange(1, len(frame_numbers) + 1, dtype=np.int32)
    ), f"Frame sequence is not continuous for {relative_path}"

loaded_path_encoder:LabelEncoder = joblib.load(DIR_OUTPUT_ENCODER)
assert np.array_equal(loaded_path_encoder.classes_, path_encoder.classes_)

print(
    f"Integrity verified: {len(loaded_frames)} frames and "
    f"{len(path_encoder.classes_)} encoded paths match the arrays in memory."
)